In [1]:
import numpy as np
import pandas as pd

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vaibhavkumar11/hindi-english-parallel-corpus")

print("Path to dataset files:", path)



Using Colab cache for faster access to the 'hindi-english-parallel-corpus' dataset.
Path to dataset files: /kaggle/input/hindi-english-parallel-corpus


In [3]:
!cp -r /kaggle/input/hindi-english-parallel-corpus /content

import pandas as pd

df = pd.read_csv("/content/hindi-english-parallel-corpus/hindi_english_parallel.csv")

df.head()

df.describe()

df = df[['english', 'hindi']]
df.dropna(inplace=True)

# Use smaller subset for speed
df = df.sample(80000, random_state=42)

df['english'] = df['english'].str.lower()
df['hindi'] = df['hindi'].str.lower()

df['hindi'] = 'start ' + df['hindi'] + ' end'

In [4]:
df.sample(4)

,english,hindi
194781,kreversi,start के-रिवर्सी end
1344303,the hard coat of plaster put by doctors to set...,start के लिए चिकित्सकों द्वारा चढ़ाई जाने वाली...
650825,enron,start एनरॉन end
35496,_ check disc…,start डिस्क जाँचें (_ c)... end


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [6]:
MAX_VOCAB = 20000

eng_tokenizer = Tokenizer(num_words=MAX_VOCAB, filters='', oov_token="<unk>")
eng_tokenizer.fit_on_texts(df['english'])

hin_tokenizer = Tokenizer(num_words=MAX_VOCAB, filters='', oov_token="<unk>")
hin_tokenizer.fit_on_texts(df['hindi'])

eng_seq = eng_tokenizer.texts_to_sequences(df['english'])
hin_seq = hin_tokenizer.texts_to_sequences(df['hindi'])

eng_vocab = MAX_VOCAB
hin_vocab = MAX_VOCAB

from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 25

encoder_input = pad_sequences(
    eng_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

decoder_input = pad_sequences(
    hin_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

# Teacher forcing target
decoder_target = np.zeros_like(decoder_input)
decoder_target[:, :-1] = decoder_input[:, 1:]
decoder_target = np.expand_dims(decoder_target, -1)

In [7]:
from sklearn.model_selection import train_test_split
X1_train, X1_val, X2_train, X2_val, y_train, y_val = train_test_split(
    encoder_input,
    decoder_input,
    decoder_target,
    test_size=0.2,
    random_state=42
)

In [8]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

In [9]:
embedding_dim = 256

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(eng_vocab, embedding_dim, mask_zero=True)(encoder_inputs)

_, state_h, state_c = LSTM(512, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(hin_vocab, embedding_dim, mask_zero=True)(decoder_inputs)

decoder_outputs, _, _ = LSTM(
    512,
    return_sequences=True,
    return_state=True
)(dec_emb, initial_state=encoder_states)

decoder_outputs = Dense(hin_vocab, activation='softmax')(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  5,120,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  5,120,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 512),     │  1,574,912 │ embedding[0][0],  │
│                     │ (None, 512),      │            │ not_equal[0][0]   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │  1,574,912 │ embedding_1[0][0… │
│                     │ 512), (None,      │            │ lstm[0][1],       │
│                     │ 512), (None,      │            │ lstm[0][2]        │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 10,260,000 │ lstm_1[0][0]      │
│                     │ 20000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,649,824 (90.22 MB)

 Trainable params: 23,649,824 (90.22 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.fit(
    [X1_train, X2_train],
    y_train,
    validation_data=([X1_val, X2_val], y_val),
    batch_size=32,
    epochs=5
)

Epoch 1/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 205s 99ms/step - accuracy: 0.2891 - loss: 6.0439 - val_accuracy: 0.2316 - val_loss: 4.9739
Epoch 2/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 200s 100ms/step - accuracy: 0.2950 - loss: 4.6991 - val_accuracy: 0.2375 - val_loss: 4.5621
Epoch 3/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 199s 100ms/step - accuracy: 0.2446 - loss: 4.0919 - val_accuracy: 0.3700 - val_loss: 4.3506
Epoch 4/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 199s 100ms/step - accuracy: 0.2728 - loss: 3.5811 - val_accuracy: 0.2597 - val_loss: 4.2632
Epoch 5/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 199s 100ms/step - accuracy: 0.3187 - loss: 3.1174 - val_accuracy: 0.2913 - val_loss: 4.2658
